In [3]:
import numpy as np

# Ramer-Douglas-Peucker Algorithm (RDP)
def rdp_algorithm(points, epsilon):
    """
    Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
    :param points: List of (x, y) tuples or numpy array of points.
    :param epsilon: The tolerance (the threshold for simplifying the polygon).
    :return: A simplified list of points.
    """
    def perpendicular_distance(point, line_start, line_end):
        # Calculate the perpendicular distance of the point from the line
        num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] + line_end[0] * line_start[1] - line_end[1] * line_start[0])
        den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
        return num / den

    def rdp_recurse(start_idx, end_idx, points, epsilon):
        # Recursively apply RDP to simplify the polygon
        if end_idx - start_idx < 2:
            return [points[start_idx], points[end_idx]]

        # Find the point with the maximum perpendicular distance
        max_distance = 0
        index = start_idx
        for i in range(start_idx + 1, end_idx):
            dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
            if dist > max_distance:
                index = i
                max_distance = dist

        # If the max distance is larger than epsilon, recurse
        if max_distance > epsilon:
            left = rdp_recurse(start_idx, index, points, epsilon)
            right = rdp_recurse(index, end_idx, points, epsilon)
            return left[:-1] + right
        else:
            return [points[start_idx], points[end_idx]]

    # Run the RDP algorithm
    return rdp_recurse(0, len(points) - 1, points, epsilon)

# Read the original .txt file
input_file = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt'
output_file = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\simplified_labels.txt'
epsilon = 0.01  # Set the simplification threshold

with open(input_file, 'r') as infile:
    lines = infile.readlines()

# Process each line in the .txt file (each polygon)
with open(output_file, 'w') as outfile:
    for line in lines:
        parts = line.strip().split()
        class_id = parts[0]  # The class ID
        points = list(map(float, parts[1:]))  # The list of coordinates
        polygon = [(points[i], points[i+1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates
        
        # Apply RDP algorithm to simplify the polygon
        simplified_polygon = rdp_algorithm(polygon, epsilon)
        
        # Flatten the simplified polygon back to a list of coordinates
        simplified_coordinates = [str(coord) for point in simplified_polygon for coord in point]
        
        # Write the simplified polygon to the output file
        outfile.write(f"{class_id} {' '.join(simplified_coordinates)}\n")

print(f"Processed labels have been saved to {output_file}.")


Processed labels have been saved to C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\simplified_labels.txt.


In [ ]:
import numpy as np

# Ramer-Douglas-Peucker Algorithm (RDP)
def rdp_algorithm(points, epsilon):
    """
    Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
    :param points: List of (x, y) tuples or numpy array of points.
    :param epsilon: The tolerance (the threshold for simplifying the polygon).
    :return: A simplified list of points.
    """
    def perpendicular_distance(point, line_start, line_end):
        # Calculate the perpendicular distance of the point from the line
        num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] +
                  line_end[0] * line_start[1] - line_end[1] * line_start[0])
        den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
        return num / den

    def rdp_recurse(start_idx, end_idx, points, epsilon):
        # Recursively apply RDP to simplify the polygon
        if end_idx - start_idx < 2:
            return [points[start_idx], points[end_idx]]

        # Find the point with the maximum perpendicular distance
        max_distance = 0
        index = start_idx
        for i in range(start_idx + 1, end_idx):
            dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
            if dist > max_distance:
                index = i
                max_distance = dist

        # If the max distance is larger than epsilon, recurse
        if max_distance > epsilon:
            left = rdp_recurse(start_idx, index, points, epsilon)
            right = rdp_recurse(index, end_idx, points, epsilon)
            return left[:-1] + right
        else:
            return [points[start_idx], points[end_idx]]

    # Run the RDP algorithm
    return rdp_recurse(0, len(points) - 1, points, epsilon)

# Function to ensure the number of points in the polygon is between 5 and 10
def adjust_polygon_points(polygon, min_points=10, max_points=20, initial_epsilon=0.01):
    epsilon = initial_epsilon
    simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    # If the polygon has too many points, simplify further
    while len(simplified_polygon) > max_points:
        epsilon *= 1.5  # Increase epsilon to simplify more
        simplified_polygon = rdp_algorithm(polygon, epsilon)

    # If the polygon has too few points, decrease epsilon and re-simplify
    while len(simplified_polygon) < min_points:
        epsilon *= 0.5  # Decrease epsilon to allow more points
        simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    return simplified_polygon

# Read the original .txt file
input_file = r'C:\Users\Spacelab3\Desktop\envs\Labeling\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt'
output_file = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt'

with open(input_file, 'r') as infile:
    lines = infile.readlines()

# Process each line in the .txt file (each polygon)
with open(output_file, 'w') as outfile:
    for line in lines:
        parts = line.strip().split()
        class_id = parts[0]  # The class ID
        points = list(map(float, parts[1:]))  # The list of coordinates
        polygon = [(points[i], points[i+1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates
        
        # Adjust the polygon to have 5 to 10 points
        adjusted_polygon = adjust_polygon_points(polygon)
        
        # Flatten the adjusted polygon back to a list of coordinates
        adjusted_coordinates = [str(coord) for point in adjusted_polygon for coord in point]
        
        # Write the adjusted polygon to the output file
        outfile.write(f"{class_id} {' '.join(adjusted_coordinates)}\n")

print(f"Processed labels have been saved to {output_file}.")


Processed labels have been saved to C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt.


In [20]:
input_file = r'C:\Users\Spacelab3\Desktop\envs\Labeling\labels\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt'

with open(input_file, 'r') as infile:
    lines = infile.readlines()
    print(lines)

['0 0.92891 0.05694 0.92812 0.05833 0.92344 0.05833 0.92266 0.05972 0.91953 0.05972 0.91719 0.06389 0.91641 0.06389 0.91406 0.06806 0.91406 0.06944 0.91250 0.07222 0.91250 0.07778 0.91172 0.07917 0.91172 0.08194 0.91094 0.08333 0.91094 0.09028 0.91016 0.09167 0.91016 0.11389 0.90938 0.11528 0.90938 0.12361 0.90859 0.12500 0.90859 0.12778 0.90781 0.12917 0.90781 0.13056 0.90703 0.13194 0.90703 0.13333 0.90625 0.13472 0.90625 0.14028 0.90703 0.14167 0.90703 0.14306 0.90938 0.14722 0.90938 0.14861 0.91016 0.15000 0.91016 0.16667 0.90938 0.16806 0.90938 0.18472 0.90859 0.18611 0.90859 0.20417 0.90938 0.20556 0.90938 0.20833 0.91016 0.20972 0.91016 0.21111 0.91328 0.21667 0.91328 0.21806 0.91406 0.21944 0.91484 0.21944 0.91641 0.22222 0.91719 0.22222 0.91797 0.22361 0.91875 0.22361 0.91953 0.22500 0.92031 0.22500 0.92109 0.22639 0.92422 0.22639 0.92500 0.22778 0.94297 0.22778 0.94375 0.22639 0.94766 0.22639 0.94844 0.22500 0.95000 0.22500 0.95078 0.22361 0.95156 0.22361 0.95312 0.22083 0.95

In [5]:
import os
import numpy as np
import logging

# Ramer-Douglas-Peucker Algorithm (RDP)
def rdp_algorithm(points, epsilon):
    """
    Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
    :param points: List of (x, y) tuples or numpy array of points.
    :param epsilon: The tolerance (the threshold for simplifying the polygon).
    :return: A simplified list of points.
    """
    def perpendicular_distance(point, line_start, line_end):
        # Calculate the perpendicular distance of the point from the line
        num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] +
                  line_end[0] * line_start[1] - line_end[1] * line_start[0])
        den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
        return num / den

    def rdp_recurse(start_idx, end_idx, points, epsilon):
        # Recursively apply RDP to simplify the polygon
        if end_idx - start_idx < 2:
            return [points[start_idx], points[end_idx]]

        # Find the point with the maximum perpendicular distance
        max_distance = 0
        index = start_idx
        for i in range(start_idx + 1, end_idx):
            dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
            if dist > max_distance:
                index = i
                max_distance = dist

        # If the max distance is larger than epsilon, recurse
        if max_distance > epsilon:
            left = rdp_recurse(start_idx, index, points, epsilon)
            right = rdp_recurse(index, end_idx, points, epsilon)
            return left[:-1] + right
        else:
            return [points[start_idx], points[end_idx]]

    # Run the RDP algorithm
    return rdp_recurse(0, len(points) - 1, points, epsilon)

# Function to ensure the number of points in the polygon is between 5 and 10
def adjust_polygon_points(polygon, min_points=10, max_points=15, initial_epsilon=0.01):
    epsilon = initial_epsilon
    simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    # If the polygon has too many points, simplify further
    while len(simplified_polygon) > max_points:
        epsilon *= 1.5  # Increase epsilon to simplify more
        simplified_polygon = rdp_algorithm(polygon, epsilon)

    # If the polygon has too few points, decrease epsilon and re-simplify
    while len(simplified_polygon) < min_points:
        epsilon *= 0.7  # Decrease epsilon to allow more points
        simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    return simplified_polygon

# Read all .txt files in a directory
def process_directory(input_dir, output_dir):
    """
    Process all .txt files in the input directory and save the results to the output directory.
    :param input_dir: Directory containing the .txt files.
    :param output_dir: Directory to save the processed files.
    """
    # List all files in the input directory
    for filename in os.listdir(input_dir):
        if filename.endswith('.txt'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            logging.info(f'{output_file_path}')
            with open(input_file_path, 'r') as infile:
                lines = infile.readlines()

            # Process the polygons and write to the output file
            with open(output_file_path, 'w') as outfile:
                for line in lines:
                    parts = line.strip().split()
                    class_id = parts[0]  # The class ID
                    points = list(map(float, parts[1:]))  # The list of coordinates
                    polygon = [(points[i], points[i + 1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates
                    
                    # Adjust the polygon to have 5 to 10 points
                    adjusted_polygon = adjust_polygon_points(polygon)
                    
                    # Flatten the adjusted polygon back to a list of coordinates
                    adjusted_coordinates = [str(coord) for point in adjusted_polygon for coord in point]
                    
                    # Write the adjusted polygon to the output file
                    outfile.write(f"{class_id} {' '.join(adjusted_coordinates)}\n")

            print(f"Processed {filename} and saved to {output_file_path}")

# Example usage
input_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests'  # Directory containing .txt files
output_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests'  # Directory to save output

# Make sure the output directory exists
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

process_directory(input_directory, output_directory)


Processed 0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
Processed 0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt
Processed 0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca85edf04c7f565d.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\Tests\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca85edf04c7f565d.txt


In [8]:
import os
import numpy as np
class PolygonSimplifier:
    def __init__(self, input_dir, output_dir, min_points=10, max_points=15, initial_epsilon=0.01):
        """
        Initialize the PolygonSimplifier object with the input directory, output directory,
        and parameters for polygon simplification.
        :param input_dir: Directory containing the .txt files.
        :param output_dir: Directory to save the processed files.
        :param min_points: Minimum number of points allowed in the simplified polygon.
        :param max_points: Maximum number of points allowed in the simplified polygon.
        :param initial_epsilon: The initial epsilon value for the RDP algorithm.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.min_points = min_points  
        self.max_points = max_points
        self.initial_epsilon = initial_epsilon

    def rdp_algorithm(self, points, epsilon):
        """
        Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
        :param points: List of (x, y) tuples or numpy array of points.
        :param epsilon: The tolerance (the threshold for simplifying the polygon).
        :return: A simplified list of points.
        """
        def perpendicular_distance(point, line_start, line_end):
            # Calculate the perpendicular distance of the point from the line
            num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] +
                      line_end[0] * line_start[1] - line_end[1] * line_start[0])
            den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
            return num / den

        def rdp_recurse(start_idx, end_idx, points, epsilon):
            # Recursively apply RDP to simplify the polygon
            if end_idx - start_idx < 2:
                return [points[start_idx], points[end_idx]]

            # Find the point with the maximum perpendicular distance
            max_distance = 0
            index = start_idx
            for i in range(start_idx + 1, end_idx):
                dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
                if dist > max_distance:
                    index = i
                    max_distance = dist

            # If the max distance is larger than epsilon, recurse
            if max_distance > epsilon:
                left = rdp_recurse(start_idx, index, points, epsilon)
                right = rdp_recurse(index, end_idx, points, epsilon)
                return left[:-1] + right
            else:
                return [points[start_idx], points[end_idx]]

        # Run the RDP algorithm
        return rdp_recurse(0, len(points) - 1, points, epsilon)

    def adjust_polygon_points(self, polygon):
        """
        Adjust the polygon to ensure the number of points is between min_points and max_points.
        :param polygon: The polygon to adjust (list of (x, y) tuples).
        :return: The adjusted polygon with a number of points between min_points and max_points.
        """
        epsilon = self.initial_epsilon
        simplified_polygon = self.rdp_algorithm(polygon, epsilon)
        
        # If the polygon has too many points, simplify further
        while len(simplified_polygon) > self.max_points:
            epsilon *= 1.5  # Increase epsilon to simplify more
            simplified_polygon = self.rdp_algorithm(polygon, epsilon)

        # If the polygon has too few points, decrease epsilon and re-simplify
        while len(simplified_polygon) < self.min_points:
            epsilon *= 0.7  # Decrease epsilon to allow more points
            simplified_polygon = self.rdp_algorithm(polygon, epsilon)
        
        return simplified_polygon

    def process_file(self, filename):
        """
        Process a single file by adjusting the polygons in it and saving the result.
        :param filename: The name of the file to process.
        :return: None
        """
        input_file_path = os.path.join(self.input_dir, filename)
        output_file_path = os.path.join(self.output_dir, os.path.split(filename)[1])

        # Read the input file and store each line of the .txt into lines
        with open(input_file_path, 'r') as infile:
            lines = infile.readlines()

        # Process the polygons and write to the output file
        with open(output_file_path, 'w') as outfile:
            for line in lines:
                parts = line.strip().split()
                class_id = parts[0]  # The class ID
                points = list(map(float, parts[1:]))  # The list of coordinates
                polygon = [(points[i], points[i + 1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates
                
                # Adjust the polygon to have between min_points and max_points
                adjusted_polygon = self.adjust_polygon_points(polygon)
                
                # Flatten the adjusted polygon back to a list of coordinates
                adjusted_coordinates = [str(coord) for point in adjusted_polygon for coord in point]
                
                # Write the adjusted polygon to the output file
                outfile.write(f"{class_id} {' '.join(adjusted_coordinates)}\n")

        print(f"Processed {filename} and saved to {output_file_path}")

    def process_directory(self):
        """
        Process all .txt files in the input directory and subdirectories, saving results to the output directory.
        :return: None
        """
        # Walk through the directory and all its subdirectories
        for dirpath, dirnames, filenames in os.walk(self.input_dir):
            for filename in filenames:
                if filename.endswith('.txt'):
                    self.process_file(os.path.relpath(os.path.join(dirpath, filename), self.input_dir))

# Example usage
input_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\testdataset'  # Directory containing .txt files
output_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\Tests'  # Directory to save output

# Make sure the output directory exists
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

process_directory(input_directory, output_directory)



In [31]:
import os
import numpy as np

# Ramer-Douglas-Peucker Algorithm (RDP)
def rdp_algorithm(points, epsilon):
    """
    Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
    :param points: List of (x, y) tuples or numpy array of points.
    :param epsilon: The tolerance (the threshold for simplifying the polygon).
    :return: A simplified list of points.
    """
    def perpendicular_distance(point, line_start, line_end):
        # Calculate the perpendicular distance of the point from the line
        num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] +
                  line_end[0] * line_start[1] - line_end[1] * line_start[0])
        den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
        return num / den

    def rdp_recurse(start_idx, end_idx, points, epsilon):
        # Recursively apply RDP to simplify the polygon
        if end_idx - start_idx < 2:
            return [points[start_idx], points[end_idx]]

        # Find the point with the maximum perpendicular distance
        max_distance = 0
        index = start_idx
        for i in range(start_idx + 1, end_idx):
            dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
            if dist > max_distance:
                index = i
                max_distance = dist

        # If the max distance is larger than epsilon, recurse
        if max_distance > epsilon:
            left = rdp_recurse(start_idx, index, points, epsilon)
            right = rdp_recurse(index, end_idx, points, epsilon)
            return left[:-1] + right
        else:
            return [points[start_idx], points[end_idx]]

    # Run the RDP algorithm
    return rdp_recurse(0, len(points) - 1, points, epsilon)

# Function to ensure the number of points in the polygon is between 5 and 10
def adjust_polygon_points(polygon, min_points=10, max_points=15, initial_epsilon=0.01):
    epsilon = initial_epsilon
    simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    # If the polygon has too many points, simplify further
    while len(simplified_polygon) > max_points:
        epsilon *= 1.5  # Increase epsilon to simplify more
        simplified_polygon = rdp_algorithm(polygon, epsilon)

    # If the polygon has too few points, decrease epsilon and re-simplify
    while len(simplified_polygon) < min_points:
        epsilon *= 0.7  # Decrease epsilon to allow more points
        simplified_polygon = rdp_algorithm(polygon, epsilon)
    
    return simplified_polygon

# Read all .txt files in a directory
def process_file(filename,input_dir):
        """
        Process a single file by adjusting the polygons in it and saving the result.
        :param filename: The name of the file to process.
        :return: None
        """
        input_file_path = os.path.join(input_dir, filename)
        
        # Read the input file and store each line of the .txt into lines
        with open(input_file_path, 'r') as infile:
            lines = infile.readlines()

        # Process the polygons and write to the output file
        with open(input_file_path, 'w') as outfile:
            for line in lines:
                parts = line.strip().split()
                class_id = parts[0]  # The class ID
                points = list(map(float, parts[1:]))  # The list of coordinates
                polygon = [(points[i], points[i + 1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates
                
                # Adjust the polygon to have between min_points and max_points
                adjusted_polygon = adjust_polygon_points(polygon)
                
                # Flatten the adjusted polygon back to a list of coordinates
                adjusted_coordinates = [str(coord) for point in adjusted_polygon for coord in point]
                
                # Write the adjusted polygon to the output file
                outfile.write(f"{class_id} {' '.join(adjusted_coordinates)}\n")

        print(f"Processed {filename} and saved to {input_file_path}")

def process_directory(input_dir):
        """
        Process all .txt files in the input directory and subdirectories, saving results to the output directory.
        :return: None
        """
        # Walk through the directory and all its subdirectories
        for dirpath, dirnames, filenames in os.walk(input_dir):
            for filename in filenames:
                if filename.endswith('.txt'):
                    process_file(os.path.relpath(os.path.join(dirpath, filename), input_dir), input_dir)
# Example usage
input_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2'  # Directory containing .txt files
output_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2'  # Directory to save output

# Make sure the output directory exists
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

process_directory(input_directory)

Processed annotations\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\annotations\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
Processed train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
Processed train\labels\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt
Processed train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca85edf04c7f565d.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca8

In [1]:
import os
import numpy as np

class PolygonProcessor:
    def __init__(self, input_dir, min_points=10, max_points=15, initial_epsilon=0.01):
        self.input_dir = input_dir
        self.min_points = min_points
        self.max_points = max_points
        self.initial_epsilon = initial_epsilon


    def rdp_algorithm(self, points, epsilon):
        """
        Apply the Ramer-Douglas-Peucker algorithm to reduce the number of points in the polygon.
        :param points: List of (x, y) tuples or numpy array of points.
        :param epsilon: The tolerance (the threshold for simplifying the polygon).
        :return: A simplified list of points.
        """
        def perpendicular_distance(point, line_start, line_end):
            # Calculate the perpendicular distance of the point from the line
            num = abs((line_end[1] - line_start[1]) * point[0] - (line_end[0] - line_start[0]) * point[1] +
                      line_end[0] * line_start[1] - line_end[1] * line_start[0])
            den = np.sqrt((line_end[1] - line_start[1]) ** 2 + (line_end[0] - line_start[0]) ** 2)
            return num / den

        def rdp_recurse(start_idx, end_idx, points, epsilon):
            # Recursively apply RDP to simplify the polygon
            if end_idx - start_idx < 2:
                return [points[start_idx], points[end_idx]]

            # Find the point with the maximum perpendicular distance
            max_distance = 0
            index = start_idx
            for i in range(start_idx + 1, end_idx):
                dist = perpendicular_distance(points[i], points[start_idx], points[end_idx])
                if dist > max_distance:
                    index = i
                    max_distance = dist

            # If the max distance is larger than epsilon, recurse
            if max_distance > epsilon:
                left = rdp_recurse(start_idx, index, points, epsilon)
                right = rdp_recurse(index, end_idx, points, epsilon)
                return left[:-1] + right
            else:
                return [points[start_idx], points[end_idx]]

        # Run the RDP algorithm
        return rdp_recurse(0, len(points) - 1, points, epsilon)

    def adjust_polygon_points(self, polygon):
        """
        Adjust the polygon to have between min_points and max_points using the RDP algorithm.
        :param polygon: List of (x, y) tuples representing the polygon points.
        :return: A simplified polygon with a number of points between min_points and max_points.
        """
        epsilon = self.initial_epsilon
        simplified_polygon = self.rdp_algorithm(polygon, epsilon)

        # If the polygon has too many points, simplify further
        while len(simplified_polygon) > self.max_points:
            epsilon *= 1.5  # Increase epsilon to simplify more
            simplified_polygon = self.rdp_algorithm(polygon, epsilon)

        # If the polygon has too few points, decrease epsilon and re-simplify
        while len(simplified_polygon) < self.min_points:
            epsilon *= 0.7  # Decrease epsilon to allow more points
            simplified_polygon = self.rdp_algorithm(polygon, epsilon)

        return simplified_polygon

    def process_file(self, filename):
        """
        Process a single file by adjusting the polygons in it and saving the result.
        :param filename: The name of the file to process.
        :return: None
        """
        input_file_path = os.path.join(self.input_dir, filename)

        # Read the input file and store each line of the .txt into lines
        with open(input_file_path, 'r') as infile:
            lines = infile.readlines()

        # Process the polygons and write to the output file
        with open(input_file_path, 'w') as outfile:
            for line in lines:
                parts = line.strip().split()
                class_id = parts[0]  # The class ID
                points = list(map(float, parts[1:]))  # The list of coordinates
                polygon = [(points[i], points[i + 1]) for i in range(0, len(points), 2)]  # Pairing x and y coordinates

                # Adjust the polygon to have between min_points and max_points
                adjusted_polygon = self.adjust_polygon_points(polygon)

                # Flatten the adjusted polygon back to a list of coordinates
                adjusted_coordinates = [str(coord) for point in adjusted_polygon for coord in point]

                # Write the adjusted polygon to the output file
                outfile.write(f"{class_id} {' '.join(adjusted_coordinates)}\n")

        print(f"Processed {filename} and saved to {input_file_path}")

    def process_directory(self):
        """
        Process all .txt files in the input directory and subdirectories, saving results to the output directory.
        :return: None
        """
        # Walk through the directory and all its subdirectories
        for dirpath, dirnames, filenames in os.walk(self.input_dir):
            for filename in filenames:
                if filename.endswith('.txt'):
                    self.process_file(os.path.relpath(os.path.join(dirpath, filename), self.input_dir))


# Example usage
input_directory = r'C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2'  # Directory containing .txt files

# Create an instance of the PolygonProcessor class
processor = PolygonProcessor(input_dir=input_directory)

# Process the directory
processor.process_directory()


Processed annotations\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\annotations\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
Processed train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_0_0_jpg.rf.03750603bf63b592779e5dabed015e4c.txt
Processed train\labels\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_1_0_jpg.rf.5889185d56e9f07d8ae724eff5448e52.txt
Processed train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca85edf04c7f565d.txt and saved to C:\Users\Spacelab3\Desktop\envs\Labeling\dataset2\train\labels\0000001_02999_d_0000005_patch_1_1_jpg.rf.fc1efc0fefd791d3ca8